# Bank Customer Estimated Salary Regression Using an Artificial Neural Network

This notebook is a practical Deep Learning implementation of an Artificial Neural Network (ANN) for regression.

The project uses the bank-customer dataset from `Churn_Modelling.csv`, but instead of predicting customer churn, the objective here is to estimate the `EstimatedSalary` value provided in the dataset.

I built this as a hands-on exercise to understand how a neural network can be used for a continuous prediction problem, starting from raw tabular data and taking it all the way through preprocessing, model training, experiment tracking, evaluation, and model persistence.

## Problem Statement

The dataset contains customer demographic, account, and financial information along with an `EstimatedSalary` field.

The objective of this project is to learn a regression function that takes a customer's profile as input and predicts their estimated salary value.

This is an educational Deep Learning project designed to understand the complete ANN regression workflow. It is not intended to represent a production payroll, compensation, or financial decision system.

### Objective

Predict:

`EstimatedSalary ∈ ℝ`

### Input Features

- Credit Score
- Geography
- Gender
- Age
- Tenure
- Balance
- Number of Products
- Has Credit Card
- Active Member Status

Identifier columns such as `RowNumber`, `CustomerId`, and `Surname` are removed because they do not represent meaningful predictive features.

The `Exited` column is also excluded from this regression task because it belongs to the separate customer-churn classification problem.

### Output

The model produces a continuous numerical estimate of `EstimatedSalary`.

The prediction is expressed in the same units used by the source dataset.

## What I am trying to understand

The main focus of this notebook is not simply getting a prediction from a neural network.

I want to understand the complete workflow:

Raw Data
→ Preprocessing
→ Feature Encoding
→ Train / Validation / Test Split
→ Feature Scaling
→ ANN Architecture
→ Training
→ Early Stopping
→ TensorBoard
→ Evaluation
→ Model Saving

The next step after this notebook is to use the saved model and preprocessing artifacts for inference through a separate prediction notebook and Streamlit application.


## 1. Import Libraries

Before starting the analysis, I am importing the libraries required for the different stages of this project.

I will use:

- **Pandas** for loading and working with tabular data
- **Scikit-learn** for preprocessing, feature scaling, and splitting the dataset
- **Pickle** for saving the preprocessing objects so that the same transformations can be reused during inference

Later in the notebook, TensorFlow/Keras will be used to build and train the Artificial Neural Network.

The idea is to keep the workflow explicit and easy to follow rather than hiding the entire process inside a single pipeline.


In [23]:
# Import Pandas for loading and manipulating tabular data
import pandas as pd

# Import train_test_split for creating training and test datasets
from sklearn.model_selection import train_test_split

# Import preprocessing tools for categorical encoding and feature scaling
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder

# Import pickle to save preprocessing objects for later inference
import pickle

## 2. Load the Dataset

I am using the bank-customer dataset from `Churn_Modelling.csv`.

The original dataset contains customer demographic, account, and financial information. Although the dataset is commonly used for churn classification, this notebook uses `EstimatedSalary` as the target so that I can study an ANN-based regression workflow.

I will first load the dataset and inspect a few rows before making any changes.


In [24]:
# Load the raw customer dataset
data = pd.read_csv(
    "/workspaces/Ann-churn-prediction/data/Churn_Modelling.csv"
)

# Display the first few records to understand the raw dataset
data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


### A Quick Look at the Dataset

Before preprocessing the data, I want to understand its size and structure.

This gives me a basic idea of how many records and variables are available before I start removing or transforming anything.


In [25]:
# Check the number of rows and columns
print(f"Dataset shape: {data.shape}")

Dataset shape: (10000, 14)


### Inspect the Available Features

The next step is to look at the column names and data types.

This helps identify identifier columns, categorical variables, numerical variables, and the target that will be used for regression.


In [26]:
# Inspect column names, data types, and non-null counts
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   RowNumber        10000 non-null  int64  
 1   CustomerId       10000 non-null  int64  
 2   Surname          10000 non-null  str    
 3   CreditScore      10000 non-null  int64  
 4   Geography        10000 non-null  str    
 5   Gender           10000 non-null  str    
 6   Age              10000 non-null  int64  
 7   Tenure           10000 non-null  int64  
 8   Balance          10000 non-null  float64
 9   NumOfProducts    10000 non-null  int64  
 10  HasCrCard        10000 non-null  int64  
 11  IsActiveMember   10000 non-null  int64  
 12  EstimatedSalary  10000 non-null  float64
 13  Exited           10000 non-null  int64  
dtypes: float64(2), int64(9), str(3)
memory usage: 1.2 MB


## 3. Remove Identifier Columns

The dataset contains a few columns that identify a customer record rather than describe the customer's characteristics.

These include:

- `RowNumber`
- `CustomerId`
- `Surname`

I am removing these columns because they are identifiers and are not intended to provide meaningful information for estimating salary.


In [27]:
# Remove identifier columns that are not intended to be model features--Preprocess the data
data = data.drop(
    ["RowNumber", "CustomerId", "Surname"],
    axis=1
)

# Confirm the remaining columns
data.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


## 4. Encode Categorical Features

Neural networks work with numerical inputs, so categorical variables need to be converted into numerical representations.

`Gender` contains two categories, so I am using `LabelEncoder` to represent them numerically.


In [28]:
# Create an encoder for the Gender column or Encode categorical variable
label_encoder_gender = LabelEncoder()

# Convert Gender categories into numerical values
data["Gender"] = label_encoder_gender.fit_transform(
    data["Gender"]
)

# Check the transformed values
data["Gender"].value_counts()

Gender
1    5457
0    4543
Name: count, dtype: int64

### One-Hot Encode Geography

`Geography` represents independent categories such as France, Germany, and Spain.

I am using one-hot encoding here because the countries do not have a natural numerical order. Creating separate binary columns prevents the model from interpreting one country as being "greater" or "less" than another.


In [29]:
# Create a one-hot encoder for Geography
one_hot_encoder_geo = OneHotEncoder(
    handle_unknown="ignore"
)

# Transform Geography into separate binary columns
geo_encoded = one_hot_encoder_geo.fit_transform(
    data[["Geography"]]
).toarray()

# Convert the encoded array back into a DataFrame
geo_encoded_df = pd.DataFrame(
    geo_encoded,
    columns=one_hot_encoder_geo.get_feature_names_out(
        ["Geography"]
    )
)

# Inspect the encoded columns
geo_encoded_df.head()

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0
1,0.0,0.0,1.0
2,1.0,0.0,0.0
3,1.0,0.0,0.0
4,0.0,0.0,1.0


### Combine the Encoded Features

The one-hot encoded Geography columns now need to be combined with the remaining customer features.

After this step, all model inputs will be represented numerically.


In [30]:
# Remove the original text-based Geography column
# and add its one-hot encoded representation
data = pd.concat(
    [
        data.drop("Geography", axis=1),
        geo_encoded_df
    ],
    axis=1
)

# Inspect the transformed dataset
data.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


## 5. Define Features and Target

Now I need to separate the variables used as model inputs from the value I want the ANN to predict.

For this notebook:

- `EstimatedSalary` is the regression target.
- `Exited` is excluded because it belongs to the separate churn classification problem.
- The remaining customer attributes become the input features.


In [31]:
# Separate the input features from the regression target
#
# EstimatedSalary is what the ANN will predict.
# Exited is intentionally excluded because it belongs
# to the separate churn classification task.
X = data.drop(
    ["EstimatedSalary", "Exited"],
    axis=1
)

# Define the continuous regression target
y = data["EstimatedSalary"]

# Check the resulting feature and target dimensions
print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")

Features shape: (10000, 11)
Target shape: (10000,)


## 6. Split the Data for Training, Validation and Testing

I want the final test set to remain untouched until the very end.

So I will use three subsets:

- **Training set** — used to learn the neural network weights
- **Validation set** — used to monitor performance during training
- **Test set** — reserved for the final evaluation

Keeping a separate test set gives me a cleaner estimate of how the trained model performs on unseen data.


In [32]:
# First create a final test set that will remain untouched during training
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

# Split the remaining training data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full,
    y_train_full,
    test_size=0.20,
    random_state=42
)

# Display the size of each split
print(f"Training set   : {X_train.shape}")
print(f"Validation set : {X_val.shape}")
print(f"Test set       : {X_test.shape}")

Training set   : (6400, 11)
Validation set : (1600, 11)
Test set       : (2000, 11)


## 7. Feature Scaling

The input features exist on very different numerical scales.

For example, Credit Score and Balance can have very different ranges. Neural networks generally train more consistently when the input features are on comparable scales.

I am therefore using `StandardScaler`.

The scaler is fitted only on the training data and then applied to the validation and test sets.


In [33]:
# Create the feature scaler
scaler = StandardScaler()

# Fit the scaler only on training data
X_train = scaler.fit_transform(X_train)

# Apply the same transformation to validation data
X_val = scaler.transform(X_val)

# Apply the same transformation to the untouched test data
X_test = scaler.transform(X_test)

## 8. Save the Preprocessing Artifacts

The model will eventually be used to make predictions on new customer data.

For inference to work correctly, new data must go through the same transformations used during training.

I am therefore saving the fitted encoders and scaler so they can be loaded later by the inference notebook and Streamlit application.


In [34]:
# Save the Gender encoder
with open("label_encoder_gender.pkl", "wb") as file:
    pickle.dump(label_encoder_gender, file)

# Save the Geography encoder
with open("one_hot_encoder_geo.pkl", "wb") as file:
    pickle.dump(one_hot_encoder_geo, file)

# Save the fitted feature scaler
with open("scaler.pkl", "wb") as file:
    pickle.dump(scaler, file)

#### ANN Regression Problem statement

## 9. Import TensorFlow and Keras

The data is now prepared and ready for modelling.

I will use TensorFlow with Keras to build a simple feed-forward Artificial Neural Network for the regression task.


In [35]:
# Import TensorFlow
import tensorflow as tf

# Import the Keras Sequential API and Input layer
from tensorflow.keras import Sequential, Input

# Import the Dense layer used to build the neural network
from tensorflow.keras.layers import Dense

## 10. Build the Artificial Neural Network

The input features are now ready for the neural network.

I am using a relatively simple architecture with two hidden layers:

```text
Input
  ↓
Dense(64, ReLU)
  ↓
Dense(32, ReLU)
  ↓
Dense(1, Linear)
```


In [36]:
# Build the ANN regression model
model = Sequential([
    # Input layer: one input value for each prepared feature
    Input(shape=(X_train.shape[1],)),

    # First hidden layer
    Dense(64, activation="relu"),

    # Second hidden layer
    Dense(32, activation="relu"),

    # Linear output layer for continuous regression
    Dense(1)
])

# Display the model architecture
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_3 (Dense)                 │ (None, 64)             │           768 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,881 (11.25 KB)

 Trainable params: 2,881 (11.25 KB)

 Non-trainable params: 0 (0.00 B)

### Compile the Model

Before training, the neural network needs an optimizer, a loss function, and evaluation metrics.

For this regression problem, I am using:

- **Adam** as the optimizer
- **Mean Absolute Error (MAE)** as the loss
- **MAE** as the main training metric

MAE measures the average absolute difference between predicted and actual salary values.


In [37]:
# Compile the regression model
model.compile(
    optimizer="adam",
    loss="mean_absolute_error",
    metrics=["mae"]
)

In [38]:
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_3 (Dense)                 │ (None, 64)             │           768 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,881 (11.25 KB)

 Trainable params: 2,881 (11.25 KB)

 Non-trainable params: 0 (0.00 B)

## 11. Set Up TensorBoard

Training a neural network is easier to understand when I can see how the model behaves over time.

TensorBoard will be used to track the training process and visualize metrics such as training and validation loss/MAE across epochs.

I am keeping the regression logs in a separate directory so they do not get mixed with the churn project's experiments.


In [39]:
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard
import datetime

# Create a separate timestamped log directory for salary regression
log_dir = (
    "logs/salary_regression/fit/"
    + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
)

# Create the TensorBoard callback
tensorboard_callback = TensorBoard(
    log_dir=log_dir,
    histogram_freq=1
)

## 12. Early Stopping

A neural network does not necessarily need to train for the maximum number of epochs.

I will monitor the validation loss during training. If the validation loss stops improving for several consecutive epochs, training can stop early.

The best model weights are restored at the end so that the final model represents the strongest validation performance observed during training.


In [40]:
# Stop training when validation loss stops improving
early_stopping_callback = EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)

## 13. Train the Model

The dataset, model architecture, optimizer, TensorBoard callback, and Early Stopping callback are now ready.

I will train the ANN using the training set while monitoring the validation set.

The final test set remains untouched and will only be used during the final evaluation stage.


In [41]:
# Train the ANN on the training data
history = model.fit(
    X_train,
    y_train,

    # Use validation data to monitor generalization
    validation_data=(X_val, y_val),

    # Maximum number of training epochs
    epochs=100,

    # Track training and stop when validation performance stops improving
    callbacks=[
        early_stopping_callback,
        tensorboard_callback
    ]
)

Epoch 1/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - loss: 100264.4922 - mae: 100264.4922 - val_loss: 100839.6016 - val_mae: 100839.6016
Epoch 2/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 99827.3438 - mae: 99827.3438 - val_loss: 99957.6406 - val_mae: 99957.6406
Epoch 3/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 98268.1875 - mae: 98268.1875 - val_loss: 97637.1562 - val_mae: 97637.1562
Epoch 4/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 95037.9609 - mae: 95037.9609 - val_loss: 93525.6094 - val_mae: 93525.6094
Epoch 5/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 89981.9531 - mae: 89981.9531 - val_loss: 87621.2969 - val_mae: 87621.2969
Epoch 6/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 83303.5078 - mae: 83303.5078 - val_loss: 80502.9766 - val_mae: 80502.9766
Epoch 7/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 75720.9766 - mae: 75720.9766 - val_loss: 72900.2578 - val_mae: 72900.2578
Epoch 8/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/ste

### Inspect the Training History

The training history contains the metrics recorded during each epoch.

I can use it to understand how the training and validation errors changed while the ANN was learning.


In [42]:
# Check which metrics were recorded during training
history.history.keys()

dict_keys(['loss', 'mae', 'val_loss', 'val_mae'])

## 14. Visualize Training with TensorBoard

TensorBoard gives me a visual view of the training process.

For this regression experiment, the main things I want to inspect are:

- Training loss / MAE
- Validation loss / MAE
- How quickly the model converged
- Whether validation performance stopped improving before the maximum number of epochs


In [43]:
%load_ext tensorboard

In [44]:
%tensorboard --logdir logs/salary_regression/fit

In [50]:
# tensorboard --logdir logs/salary_regression/fit --host 0.0.0.0 --port 6007

## 15. Final Model Evaluation

The model has finished training and the final test set has remained untouched throughout the training process.

I can now evaluate the trained ANN on this unseen test data to measure its final regression performance.


In [45]:
# Evaluate the trained model on the final untouched test set
test_loss, test_mae = model.evaluate(
    X_test,
    y_test,
    verbose=0
)

print(f"Test MAE: {test_mae:,.2f}")

Test MAE: 50,329.83


### Additional Regression Metrics

MAE is useful, but one metric alone does not describe the complete behaviour of a regression model.

I will also calculate:

- **RMSE** — gives more weight to larger errors
- **R²** — measures how much of the variation in the target is explained by the model

These metrics provide additional context for the final evaluation.


In [46]:
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

# Generate predictions for the final test set
y_pred = model.predict(
    X_test,
    verbose=0
).ravel()

# Calculate MAE
mae = np.mean(
    np.abs(y_test - y_pred)
)

# Calculate RMSE
rmse = np.sqrt(
    mean_squared_error(y_test, y_pred)
)

# Calculate R²
r2 = r2_score(
    y_test,
    y_pred
)

print(f"MAE  : {mae:,.2f}")
print(f"RMSE : {rmse:,.2f}")
print(f"R²   : {r2:.4f}")

MAE  : 50,329.84
RMSE : 58,256.71
R²   : -0.0280


## 16. Compare Actual and Predicted Values

A metric gives me a numerical summary, but it is also useful to look at some individual predictions.

I will compare the actual salary values from the test set with the values predicted by the ANN.


In [47]:
# Create a small comparison table
comparison = pd.DataFrame({
    "Actual Salary": y_test.values,
    "Predicted Salary": y_pred
})

# Display a sample of predictions
comparison.head(10)

,Actual Salary,Predicted Salary
0,41788.37,85882.648438
1,146379.30,92043.445312
2,58561.31,99791.562500
3,170679.74,102518.859375
4,114669.79,106940.023438
5,149418.41,92404.437500
6,75685.97,106004.515625
7,70529.00,104787.046875
8,16618.76,98719.218750
9,164104.74,98408.804688


## 16A. Compare Against a Simple Baseline

Before judging the neural network, I want to compare it with a very simple baseline.

A baseline that always predicts the typical salary value gives us a useful reference point. If the ANN cannot improve meaningfully over this baseline, the issue may not be the neural network architecture itself — the available input features may simply contain limited information about the target.


In [51]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Use the median salary from the training set as a simple baseline
baseline_prediction = np.full(
    shape=len(y_test),
    fill_value=y_train.median()
)

# Evaluate the baseline
baseline_mae = mean_absolute_error(
    y_test,
    baseline_prediction
)

baseline_rmse = np.sqrt(
    mean_squared_error(y_test, baseline_prediction)
)

baseline_r2 = r2_score(
    y_test,
    baseline_prediction
)

print(f"Baseline MAE  : {baseline_mae:,.2f}")
print(f"Baseline RMSE : {baseline_rmse:,.2f}")
print(f"Baseline R²   : {baseline_r2:.4f}")

Baseline MAE  : 49,776.24
Baseline RMSE : 57,471.51
Baseline R²   : -0.0005


In [52]:
y.describe()

count     10000.000000
mean     100090.239881
std       57510.492818
min          11.580000
25%       51002.110000
50%      100193.915000
75%      149388.247500
max      199992.480000
Name: EstimatedSalary, dtype: float64

In [53]:
data.select_dtypes(include="number").corr()["EstimatedSalary"].sort_values(
    ascending=False
)

EstimatedSalary      1.000000
NumOfProducts        0.014204
Balance              0.012797
Exited               0.012097
Geography_Germany    0.010297
Tenure               0.007784
CreditScore         -0.001384
Geography_France    -0.003332
Geography_Spain     -0.006482
Age                 -0.007201
Gender              -0.008112
HasCrCard           -0.009933
IsActiveMember      -0.011421
Name: EstimatedSalary, dtype: float64

## 17. Save the Trained Model

The model has now been trained and evaluated.

I will save the trained ANN so that it can be reused later without retraining it from scratch.

I am using Keras' native `.keras` format for the saved model.


In [48]:
# Save the trained ANN in Keras' native format
model.save(
    "salary_regression_model.keras"
)

## 18. What I Completed

At this stage, the complete ANN regression workflow is finished.

Starting from the raw customer dataset, I:

1. Loaded and inspected the data
2. Removed identifier columns
3. Encoded categorical features
4. Separated the regression target from the input features
5. Created training, validation, and test sets
6. Scaled the numerical inputs
7. Saved the preprocessing objects
8. Built and compiled an ANN
9. Trained it with Early Stopping
10. Tracked the experiment with TensorBoard
11. Evaluated the model on unseen test data
12. Saved the trained model for later inference

The next step is to load this saved model and preprocessing artifacts in a separate inference notebook and eventually connect the same prediction pipeline to a Streamlit application.


## 19. What the Results Tell Me

The ANN was able to complete the regression workflow successfully, but the final evaluation showed that the model did not learn a useful relationship between the available customer features and `EstimatedSalary`.

The test MAE was approximately 50K, while a simple median-based baseline achieved a slightly lower error. The resulting R² was also slightly below zero.

The feature-level correlation analysis showed that the available variables had very weak linear relationships with `EstimatedSalary`.

This suggests that the main limitation is not simply the neural network architecture. The available features provide limited predictive signal for the target used in this experiment.

This was an important practical learning outcome: model complexity cannot compensate for a target that is poorly supported by the available features.

For a production-oriented salary prediction application, I would use a dataset containing features that are directly relevant to compensation and retrain the regression pipeline on that data.
